# ❄️🐉 cryoDRGN — back-project particle subsets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ts387/cryodrgn/blob/claude/cryodrgn-colab-notebook-5mf3p3/cryoDRGN_colab_backproject.ipynb)

Reconstruct a **separate volume for each of several particle subsets** with
`cryodrgn backproject_voxel --ind`, then compare them.

Give it a folder of index files — one per subset — and it back-projects each in turn,
resuming where it left off if the session drops. Every reconstruction also yields two
half-maps and a cryoSPARC-style FSC curve, so each subset arrives with its own
resolution estimate.

| Step | What happens |
|------|--------------|
| 1. Setup | Check the GPU, install cryoDRGN, mount Drive |
| 2. Inputs | Particle stack, `pose.pkl`, `ctf.pkl` — the same files you trained with |
| 3. Subsets | Load and validate the index sets; optionally equalise their sizes |
| 4. Back-project | One volume per subset, resumable, mirrored to Drive |
| 5. Compare | Resolution table, pairwise FSC matrix, slice montage, difference maps |
| 6. Export | Zip the lot back to Drive |

> **You do not need a trained model here.** Back-projection uses only the particles, the
> poses and the CTF — the same `pose.pkl` and `ctf.pkl` you fed to `train_vae`. The subsets
> themselves are usually *derived* from a cryoDRGN latent (k-means labels, an occupancy
> percentile cut, UMAP lobes), but this notebook doesn't care where they came from.

> **Index conventions.** Indices are **0-based positions into the particle stack**, exactly as
> `cryodrgn analyze` / `filter` emit them. If you trained with `--ind`, indices derived from
> `z.N.pkl` are positions within *that subset*, not the full stack — compose them first with
> `cryodrgn_utils select_clusters --parent-ind`.

## 1 · Setup

In [ ]:
#@title 1.1 · Check the GPU runtime { display-mode: "form" }
#@markdown Confirms a CUDA GPU is attached. If this prints **"No GPU found"**, go to
#@markdown **Runtime → Change runtime type → GPU** and re-run this cell.
import subprocess, sys

print("=" * 60)
gpu = subprocess.run(["nvidia-smi",
                      "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode == 0 and gpu.stdout.strip():
    name, mem, driver = [x.strip() for x in gpu.stdout.strip().split(",")]
    print(f"✅ GPU detected : {name}")
    print(f"   Memory       : {mem}")
    print(f"   Driver       : {driver}")
else:
    print("❌ No GPU found!")
    print("   Runtime → Change runtime type → Hardware accelerator → GPU,")
    print("   then re-run this cell. cryoDRGN training needs a GPU.")
print("=" * 60)

In [ ]:
#@title 1.2 · Install cryoDRGN { display-mode: "form" }
#@markdown Installs cryoDRGN from PyPI. Colab's pre-installed PyTorch/CUDA are kept.
#@markdown <br>• **stable** – the recommended release &nbsp;•&nbsp; **beta** – newest dev build from TestPyPI
release_channel = "stable"  #@param ["stable", "beta"]
#@markdown Optionally pin an exact version (e.g. `4.3.0`); leave blank for the latest.
version = ""  #@param {type:"string"}
#@markdown A few dependencies are pinned to versions other than Colab's defaults, so the
#@markdown runtime **restarts automatically** at the end. That is expected — just carry
#@markdown on with the next cell afterwards.
restart_after_install = True  #@param {type:"boolean"}

import subprocess, sys

pkg = "cryodrgn"
if version.strip():
    pkg = f"cryodrgn=={version.strip()}"

if release_channel == "beta":
    cmd = [sys.executable, "-m", "pip", "install", "-q",
           "-i", "https://test.pypi.org/simple/",
           "--extra-index-url", "https://pypi.org/simple/",
           "cryodrgn", "--pre"]
    if version.strip():
        cmd[cmd.index("cryodrgn")] = pkg
else:
    cmd = [sys.executable, "-m", "pip", "install", "-q", pkg]

print("Installing", pkg, f"({release_channel} channel) — this takes ~1-2 min...\n")
ret = subprocess.run(cmd)
if ret.returncode != 0:
    raise SystemExit("❌ pip install failed — see the log above.")

# --- realign torchvision with torch -------------------------------------------------
# cryoDRGN pins torch<2.10, so pip may DOWNGRADE Colab's torch. Colab's pre-installed
# torchvision was compiled against the newer torch, and once they disagree importing it
# raises "operator torchvision::nms does not exist". That breaks EVERY cryodrgn command,
# because the CLI eagerly imports all command modules and analyze_landscape_full imports
# umap -> torchvision. Matching pair is torch 2.N <-> torchvision 0.(N+15).
import importlib.metadata as md

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

tver, tvver = _ver("torch"), _ver("torchvision")
if tver and tvver:
    tmaj, tmin = (int(x) for x in tver.split(".")[:2])
    tvmin = int(tvver.split(".")[1])
    want = tmin + 15 if tmaj == 2 else None
    if want is not None and tvmin != want:
        print(f"\n⚠️  torch {tver} and torchvision {tvver} are incompatible "
              f"(cryoDRGN's torch<2.10 pin downgraded torch).")
        print(f"   Installing torchvision 0.{want}.* to match...")
        fix = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--no-deps",
             f"torchvision==0.{want}.*"])
        if fix.returncode == 0:
            print(f"   ✅ torchvision realigned to 0.{want}.*")
        else:
            print(f"   ❌ Could not install torchvision 0.{want}.* — if cryodrgn commands "
                  f"fail with 'torchvision::nms does not exist', run:")
            print(f"      !pip install --no-deps 'torchvision==0.{want}.*'")

print("\n✅ cryoDRGN installed.")
if restart_after_install:
    print("🔄 Restarting the runtime to finalize the install (this is normal)...")
    print("   When it reconnects, continue from cell 1.3 — do NOT re-run this cell.")
    get_ipython().kernel.do_shutdown(True)

In [ ]:
#@title 1.3 · Verify the installation { display-mode: "form" }
#@markdown Run this **after** the runtime has restarted. The `cryodrgn --version` smoke-test is
#@markdown the important one: the CLI imports *every* command module on startup, so a broken
#@markdown dependency anywhere makes all commands fail — better to catch it here than mid-run.
import sys, subprocess
import torch, cryodrgn

print(f"cryoDRGN version : {cryodrgn.__version__}")
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device      : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  CUDA not available — fine for Step 4 (CPU-only), but Step 6 needs a GPU (cell 1.1).")

# torchvision must match torch or `import umap` blows up inside the cryodrgn CLI
try:
    import torchvision
    print(f"torchvision      : {torchvision.__version__} (ok)")
except Exception as e:
    msg = str(e).splitlines()[0]
    want = "0.%d.*" % (int(torch.__version__.split(".")[1]) + 15)
    if "numpy.dtype size changed" in msg or "binary incompatibility" in msg:
        # cryoDRGN pins numpy<1.27, downgrading Colab's numpy 2.x. C extensions that were
        # compiled against numpy 2.x headers then fail their ABI check on import.
        print(f"⚠️  torchvision fails a NumPy ABI check: {msg}")
        print("   Cause: cryoDRGN pins numpy<1.27, so Colab's numpy 2.x was downgraded and")
        print("   torchvision (built against numpy 2.x) no longer matches.")
        print("   This is USUALLY HARMLESS: cryoDRGN never imports torchvision itself — only")
        print("   umap does, and cryodrgn.analysis imports umap lazily. Every cryodrgn command")
        print("   also runs as a subprocess. Treat the CLI check below as the real verdict, and")
        print("   don't 'fix' this unless something actually fails.")
    else:
        print(f"❌ torchvision is broken: {msg}")
        print("   Looks like a torch/torchvision version mismatch rather than a NumPy issue.")
        print(f"   Fix with:  !pip install --no-deps 'torchvision=={want}'")
        print("   then re-run this cell (no restart needed).")

print("\n$ cryodrgn --version")
r = subprocess.run(["cryodrgn", "--version"], capture_output=True, text=True)
print((r.stdout + r.stderr).strip()[-2000:])
if r.returncode != 0:
    raise RuntimeError("The cryodrgn CLI failed to start — fix the error above before continuing.")
print("\n✅ CLI healthy — all command modules import cleanly.")

In [ ]:
#@title 1.4 · Mount Google Drive { display-mode: "form" }
#@markdown Click the link that appears, pick your Google account, and paste the code
#@markdown (or approve the pop-up). Your Drive appears under `/content/drive/MyDrive`.
from google.colab import drive
drive.mount("/content/drive")
print("\n✅ Drive mounted at /content/drive/MyDrive")

## 2 · Inputs

Point at the **same** stack, poses and CTF you used for training. Two things matter here and
both are easy to get wrong:

- **`uninvert_data` must match training.** `backproject_voxel` has its own copy of the flag with
  the same default as `train_vae`; getting it wrong inverts the map's contrast.
- **The stack should be on local disk, not Drive.** Back-projection seeks to each selected
  particle individually, and Drive's FUSE mount is ~30× slower than local disk for that access
  pattern. Cell 2.2 checks and offers to copy.

Pixel size is handled for you — `backproject_voxel` reads it from `ctf.pkl` and rescales it to
the current box (`ctf.py:160-162`), so a downsampled stack gets the right Å/px automatically.

In [ ]:
#@title 2.1 · Project folder & input files { display-mode: "form" }
#@markdown Your Drive project folder — the same one the main notebook used.
drive_project_dir = "/content/drive/MyDrive/cryodrgn_project"  #@param {type:"string"}
#@markdown Fast local scratch.
local_work_dir = "/content/cryodrgn_work"  #@param {type:"string"}
#@markdown Particle stack. Leave blank to auto-detect a `*.mrcs`/`*.txt` in the project folder.
particles = ""  #@param {type:"string"}
#@markdown Poses and CTF (blank = `pose.pkl` / `ctf.pkl` in the project folder).
pose_pkl = ""  #@param {type:"string"}
ctf_pkl = ""  #@param {type:"string"}
#@markdown Needed only for `.star`/`.cs` inputs that hold relative paths.
datadir = ""  #@param {type:"string"}
#@markdown **Must match training.** Tick if you trained with `--uninvert-data`.
uninvert_data = True  #@param {type:"boolean"}

import os, glob

DRIVE_DIR = os.path.abspath(drive_project_dir)
WORK_DIR = os.path.abspath(local_work_dir)
os.makedirs(WORK_DIR, exist_ok=True)
if not os.path.isdir(DRIVE_DIR):
    raise FileNotFoundError(f"{DRIVE_DIR} not found — check drive_project_dir (and run 1.4).")

def _pick(explicit, *patterns):
    if explicit.strip():
        return os.path.abspath(explicit.strip())
    for pat in patterns:
        hits = sorted(glob.glob(os.path.join(DRIVE_DIR, pat)))
        hits += sorted(glob.glob(os.path.join(DRIVE_DIR, "*", pat)))
        if hits:
            return hits[0]
    return ""

parts = _pick(particles, "*.mrcs", "*.txt", "*.star", "*.cs")
pose = _pick(pose_pkl, "pose.pkl")
ctf = _pick(ctf_pkl, "ctf.pkl")

missing = [n for n, p in (("particles", parts), ("pose.pkl", pose), ("ctf.pkl", ctf)) if not p]
if missing:
    raise FileNotFoundError(
        f"Could not locate: {', '.join(missing)}. Fill the fields above explicitly.\n"
        f"Looked in {DRIVE_DIR} and one level below."
    )
for n, p in (("particles", parts), ("pose.pkl", pose), ("ctf.pkl", ctf)):
    if not os.path.exists(p):
        raise FileNotFoundError(f"{n}: {p} does not exist.")

os.environ["BP_DRIVE_DIR"] = DRIVE_DIR
os.environ["BP_WORK_DIR"] = WORK_DIR
os.environ["BP_PARTICLES"] = parts
os.environ["BP_POSE"] = pose
os.environ["BP_CTF"] = ctf
os.environ["BP_DATADIR"] = datadir.strip()
os.environ["BP_UNINVERT"] = "1" if uninvert_data else "0"
os.chdir(WORK_DIR)

print(f"📁 Drive project : {DRIVE_DIR}")
print(f"⚡ Local scratch : {WORK_DIR}")
print(f"🧊 particles     : {parts}  ({os.path.getsize(parts)/2**30:.2f} GB)")
print(f"📐 poses         : {pose}")
print(f"🔬 ctf           : {ctf}")
print(f"🔄 uninvert-data : {'ON  (--uninvert-data will be passed)' if uninvert_data else 'off'}")
if datadir.strip():
    print(f"📂 datadir       : {datadir.strip()}")

In [ ]:
#@title 2.2 · Stage the stack on local disk (strongly recommended) { display-mode: "form" }
#@markdown Back-projection seeks to each selected particle individually. On Drive's FUSE mount
#@markdown that is roughly 30x slower than local disk, and this notebook reads the stack once
#@markdown per subset. Copying it up front usually pays for itself after the second subset.
#@markdown Untick to back-project in place (fine if the stack is already local, or tiny).
copy_to_local = True  #@param {type:"boolean"}

import os, shutil, time

parts = os.environ["BP_PARTICLES"]
WORK_DIR = os.environ["BP_WORK_DIR"]
on_drive = parts.startswith("/content/drive")

def _members(txt_path):
    """The .mrcs files a .txt stack indexes, resolved against the .txt's own directory."""
    base = os.path.dirname(os.path.abspath(txt_path))
    with open(txt_path) as f:
        rels = [ln.strip() for ln in f if ln.strip()]
    return rels, base

def _stage_txt(txt_path, dest_dir):
    """Copy a .txt stack AND the .mrcs it references, preserving the relative layout.

    Copying the .txt alone is useless: it lists basenames resolved against its own
    directory, so a lone .txt in a new folder points at nothing."""
    rels, base = _members(txt_path)
    total = sum(os.path.getsize(os.path.join(base, r)) for r in rels)
    free = shutil.disk_usage(dest_dir).free
    print(f"  .txt stack indexes {len(rels)} file(s), {total/2**30:.1f} GiB total")
    if free < total * 1.05:
        raise RuntimeError(
            f"Need {total/2**30:.1f} GiB on {dest_dir} but only {free/2**30:.1f} GiB free. "
            f"Untick copy_to_local to read from Drive instead (much slower), or free space."
        )
    for i, r in enumerate(rels, 1):
        s_, d_ = os.path.join(base, r), os.path.join(dest_dir, r)
        os.makedirs(os.path.dirname(d_) or dest_dir, exist_ok=True)
        if os.path.exists(d_) and os.path.getsize(d_) == os.path.getsize(s_):
            print(f"  [{i}/{len(rels)}] {r} already staged", flush=True)
            continue
        t0 = time.time()
        shutil.copyfile(s_, d_)
        print(f"  [{i}/{len(rels)}] {r}  {os.path.getsize(d_)/1e9:.1f} GB in "
              f"{time.time()-t0:.0f}s", flush=True)
    out_txt = os.path.join(dest_dir, os.path.basename(txt_path))
    shutil.copyfile(txt_path, out_txt)
    return out_txt

if not on_drive:
    print(f"✔ Already on local disk: {parts}")
elif not copy_to_local:
    print(f"⚠️  Stack is on Drive and copy_to_local is off — expect slow back-projection:\n   {parts}")
elif parts.endswith(".txt"):
    stack_dir = os.path.join(WORK_DIR, "stack")
    os.makedirs(stack_dir, exist_ok=True)
    dst = _stage_txt(parts, stack_dir)
    os.environ["BP_PARTICLES"] = dst
    print(f"✔ Using {dst}")
else:
    size_gb = os.path.getsize(parts) / 2**30
    free_gb = shutil.disk_usage(WORK_DIR).free / 2**30
    if free_gb < size_gb * 1.05:
        raise RuntimeError(
            f"Need {size_gb:.1f} GB on {WORK_DIR} but only {free_gb:.1f} GB free. "
            f"Untick copy_to_local to read from Drive instead, or free some space."
        )
    dst = os.path.join(WORK_DIR, os.path.basename(parts))
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(parts):
        print(f"✔ Local copy already complete: {dst}")
    else:
        print(f"Copying {size_gb:.2f} GB → {dst} ...")
        t0 = time.time()
        shutil.copyfile(parts, dst)
        dt = time.time() - t0
        print(f"✔ Done in {dt/60:.1f} min ({size_gb/max(dt,1e-9)*1024:.0f} MB/s)")
    os.environ["BP_PARTICLES"] = dst
    print(f"Using {dst}")

## 3 · The index sets

Cell 3.1 takes a **folder or `.zip`** and discovers every index file in it. Two formats are
accepted, and both are normalised to the pickled integer arrays `backproject_voxel` expects
(`--ind` is read with `utils.load_pkl`, so a bare `.txt` will not work as-is):

- **`.pkl`** — a pickled NumPy integer array (what `cryodrgn analyze`/`filter` write)
- **`.txt`** — one 0-based index per line

When a `.pkl` and a `.txt` share a stem, the `.pkl` wins. macOS `__MACOSX` resource forks and
dot-underscore files are ignored.

It then validates everything against the stack: dtype, bounds, duplicates within a set,
overlap between sets, and total coverage.

In [ ]:
#@title 3.1 · Load & validate index sets { display-mode: "form" }
#@markdown Folder **or** `.zip` holding the index files. Blank = look for
#@markdown `indices/` or any `*index*.zip` in the Drive project folder.
indices_source = ""  #@param {type:"string"}
#@markdown Optional glob to keep only some sets, e.g. `*lobe0[19]*` for lobes 1 and 9.
name_filter = "*"  #@param {type:"string"}

import os, re, glob, pickle, zipfile, fnmatch
import numpy as np
from cryodrgn import utils
from cryodrgn.source import ImageSource

DRIVE_DIR = os.environ["BP_DRIVE_DIR"]
WORK_DIR = os.environ["BP_WORK_DIR"]

src = indices_source.strip()
if not src:
    for cand in [os.path.join(DRIVE_DIR, "indices")] + \
                sorted(glob.glob(os.path.join(DRIVE_DIR, "*index*.zip"))) + \
                sorted(glob.glob(os.path.join(DRIVE_DIR, "*ind*.zip"))):
        if os.path.exists(cand):
            src = cand
            break
if not src or not os.path.exists(src):
    raise FileNotFoundError(
        "No index source found. Put your .pkl/.txt index files in a folder (or a .zip) "
        "and give its path above."
    )

# unpack a zip into local scratch so we never write to Drive
if src.lower().endswith(".zip"):
    dest = os.path.join(WORK_DIR, "index_sets")
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(src) as zf:
        zf.extractall(dest)
    print(f"Unzipped {src} → {dest}")
    search_root = dest
else:
    search_root = src

found = []
for root, dirs, files in os.walk(search_root):
    dirs[:] = [d for d in dirs if d != "__MACOSX"]          # macOS resource forks
    for fn in files:
        if fn.startswith("._") or not fn.lower().endswith((".pkl", ".txt")):
            continue
        found.append(os.path.join(root, fn))

# a .pkl and .txt with the same stem are the same set; prefer the .pkl
by_stem = {}
for p in sorted(found):
    stem = re.sub(r"^(ind_|indices_|particles_)", "", os.path.splitext(os.path.basename(p))[0])
    if stem not in by_stem or p.lower().endswith(".pkl"):
        by_stem[stem] = p
by_stem = {k: v for k, v in by_stem.items() if fnmatch.fnmatch(k, name_filter)}
if not by_stem:
    raise FileNotFoundError(f"No .pkl/.txt index files under {search_root} matching '{name_filter}'.")

# how many particles are in the stack? (header read only — does not load the images)
n_stack = ImageSource.from_file(os.environ["BP_PARTICLES"],
                                lazy=True,
                                datadir=os.environ["BP_DATADIR"] or None).n
print(f"Particle stack holds {n_stack:,} images\n")

norm_dir = os.path.join(WORK_DIR, "index_sets_pkl")
os.makedirs(norm_dir, exist_ok=True)

sets, rows = {}, []
for stem, path in sorted(by_stem.items()):
    if path.lower().endswith(".pkl"):
        idx = np.asarray(utils.load_pkl(path))
    else:
        idx = np.loadtxt(path, dtype=np.int64, ndmin=1)
    if idx.dtype == bool:               # accept a boolean mask too
        idx = np.where(idx)[0]
    idx = np.asarray(idx).ravel().astype(np.int64)

    problems = []
    if idx.size == 0:
        problems.append("EMPTY")
    if idx.size and idx.min() < 0:
        problems.append(f"negative index {int(idx.min())}")
    if idx.size and idx.max() >= n_stack:
        problems.append(f"index {int(idx.max())} >= stack size {n_stack}")
    dupes = idx.size - np.unique(idx).size
    if dupes:
        problems.append(f"{dupes} duplicate(s)")
    if problems:
        raise ValueError(f"{os.path.basename(path)}: " + "; ".join(problems))

    # backproject_voxel reads --ind with utils.load_pkl, so normalise everything to .pkl
    out_pkl = os.path.join(norm_dir, f"{stem}.pkl")
    utils.save_pkl(np.sort(idx), out_pkl)
    sets[stem] = out_pkl
    rows.append((stem, idx.size, int(idx.min()), int(idx.max()), os.path.basename(path)))

w = max(len(r[0]) for r in rows)
print(f"{'set'.ljust(w)}  {'N':>9}  {'min':>8}  {'max':>8}   source")
print("-" * (w + 46))
for stem, n, lo, hi, srcname in rows:
    print(f"{stem.ljust(w)}  {n:>9,}  {lo:>8,}  {hi:>8,}   {srcname}")

allidx = np.concatenate([np.asarray(utils.load_pkl(p)) for p in sets.values()])
uniq = np.unique(allidx)
print("-" * (w + 46))
print(f"{'TOTAL'.ljust(w)}  {allidx.size:>9,}")
print(f"\nunique particles covered : {uniq.size:,} of {n_stack:,} "
      f"({uniq.size/n_stack*100:.2f}%)")
overlap = allidx.size - uniq.size
print(f"overlap between sets     : {overlap:,} "
      f"{'(sets are disjoint)' if overlap == 0 else '(sets SHARE particles)'}")
print(f"unassigned particles     : {n_stack - uniq.size:,}")

sizes = np.array([r[1] for r in rows])
print(f"\nsize range               : {sizes.min():,} … {sizes.max():,} "
      f"({sizes.max()/max(sizes.min(),1):.1f}x)")
if sizes.max() / max(sizes.min(), 1) > 1.5:
    print("⚠️  Subsets differ substantially in size, so their maps will differ in SNR and\n"
          "    apparent resolution for reasons that have nothing to do with structure.\n"
          "    Run 3.2 to equalise them before drawing structural conclusions.")

import json as _json
os.environ["BP_SETS"] = _json.dumps(sets)
os.environ["BP_NSTACK"] = str(n_stack)

In [ ]:
#@title 3.2 · (Recommended) Equalise subset sizes { display-mode: "form" }
#@markdown A map from 117,000 particles will out-resolve one from 8,000 no matter what is in
#@markdown them, so an unequalised comparison mostly measures subset size. This draws a random
#@markdown sample of the same size from every set, writing a parallel `*_eqN` set.
#@markdown
#@markdown `0` = use the size of the smallest set. Sets already at or below the target are kept whole.
target_n = 0  #@param {type:"integer"}
#@markdown Seed for the draw, so the selection is reproducible.
random_seed = 0  #@param {type:"integer"}
#@markdown Untick to skip and back-project the sets at their native sizes.
apply_equalisation = True  #@param {type:"boolean"}

import os, json
import numpy as np
from cryodrgn import utils

sets = json.loads(os.environ["BP_SETS"])
if not apply_equalisation:
    print("Skipped — back-projecting the sets at their native sizes.")
    print("Remember that resolution differences between them will be dominated by particle count.")
else:
    WORK_DIR = os.environ["BP_WORK_DIR"]
    norm_dir = os.path.join(WORK_DIR, "index_sets_pkl")
    loaded = {k: np.asarray(utils.load_pkl(v)) for k, v in sets.items()}
    n_target = int(target_n) if int(target_n) > 0 else min(len(v) for v in loaded.values())

    rng = np.random.default_rng(int(random_seed))
    eq = {}
    print(f"Equalising to N = {n_target:,} (seed {int(random_seed)})\n")
    for k, v in sorted(loaded.items()):
        if len(v) <= n_target:
            keep = v
            note = "kept whole"
        else:
            keep = np.sort(rng.choice(v, size=n_target, replace=False))
            note = f"sampled from {len(v):,}"
        out = os.path.join(norm_dir, f"{k}_eq{n_target}.pkl")
        utils.save_pkl(keep, out)
        eq[f"{k}_eq{n_target}"] = out
        print(f"  {k:<22} {len(keep):>8,}   ({note})")

    os.environ["BP_SETS"] = json.dumps(eq)
    print(f"\n✅ {len(eq)} equalised sets will be back-projected.")
    print("   The native-size .pkl files are still in", norm_dir)

## 4 · Back-project every subset

One `cryodrgn backproject_voxel` per set. Each writes into its own folder:

```
backproject/<set>/backproject.mrc     full reconstruction
                 half_map_a.mrc       \  interleaved halves
                 half_map_b.mrc       /
                 fsc-vals.txt         FSC curves: NoMask / Spherical / Loose / Tight / Corrected
                 fsc-plot.png         the same, plotted
```

The cell is **resumable**: a set whose `backproject.mrc` already exists is skipped, so if the
session drops you can simply re-run it. Results are mirrored to Drive as each set completes,
which is what makes that safe across a disconnect.

In [ ]:
#@title 4.1 · Run back-projection for all sets { display-mode: "form" }
#@markdown Images per batch. Lower this if you hit CUDA OOM on a large box.
batch_size = 8  #@param [4, 8, 16, 32] {type:"raw"}
#@markdown Stream images from disk instead of loading each subset into RAM. `auto` turns it on
#@markdown per subset when the eager footprint (N x D x D x 4 bytes) would not fit in RAM —
#@markdown at D=256 a 300k-particle subset is 79 GB, which no Colab runtime holds.
lazy = "auto"  #@param ["auto", "always", "never"]
#@markdown Quick smoke test: cap each subset at this many particles (`0` = use all).
first_n = 0  #@param {type:"integer"}
#@markdown Regularisation added to the weight map (cryoDRGN default is 1.0).
reg_weight = 1.0  #@param {type:"number"}
#@markdown Recompute sets that already have a `backproject.mrc`.
overwrite = False  #@param {type:"boolean"}

import os, json, glob, shutil, time
import numpy as np
from cryodrgn import utils
from cryodrgn.source import ImageSource
sets = json.loads(os.environ["BP_SETS"])
parts, pose, ctf = (os.environ[k] for k in ("BP_PARTICLES", "BP_POSE", "BP_CTF"))
WORK_DIR, DRIVE_DIR = os.environ["BP_WORK_DIR"], os.environ["BP_DRIVE_DIR"]

bp_root = os.path.join(WORK_DIR, "backproject")
drive_root = os.path.join(DRIVE_DIR, "backproject")
os.makedirs(bp_root, exist_ok=True)
os.makedirs(drive_root, exist_ok=True)
os.environ["BP_ROOT"] = bp_root
os.environ["BP_DRIVE_ROOT"] = drive_root

def _done(d):
    return os.path.exists(os.path.join(d, "backproject.mrc"))

# How much RAM would each subset need if loaded eagerly? backproject_voxel builds its dataset
# with lazy=args.lazy, and a non-lazy ImageSource materialises the whole filtered stack as
# float32 (source.py:116-126).
_D = ImageSource.from_file(parts, lazy=True, datadir=os.environ["BP_DATADIR"] or None).D
try:
    _avail = os.sysconf("SC_AVPHYS_PAGES") * os.sysconf("SC_PAGE_SIZE")
except (ValueError, OSError):
    _avail = 8e9
_budget = _avail * 0.7          # leave room for torch, the volume accumulators and the OS
print(f"box {_D}x{_D}; {_avail/1e9:.0f} GB RAM available, "
      f"eager subsets budgeted to {_budget/1e9:.0f} GB\n")

def _eager_gb(n):
    return n * _D * _D * 4 / 1e9

todo = []
for name, ind_pkl in sorted(sets.items()):
    outdir = os.path.join(bp_root, name)
    # a completed run may only exist on Drive after a session restart — pull it back
    if not _done(outdir) and _done(os.path.join(drive_root, name)):
        shutil.copytree(os.path.join(drive_root, name), outdir, dirs_exist_ok=True)
        print(f"restored {name} from Drive")
    if _done(outdir) and not overwrite:
        print(f"⏭  {name}: already reconstructed — skipping")
        continue
    todo.append((name, ind_pkl, outdir))

print(f"\n{len(todo)} of {len(sets)} set(s) to run\n" + "=" * 70)
t_all = time.time()
for i, (name, ind_pkl, outdir) in enumerate(todo, 1):
    cmd = (f'cryodrgn backproject_voxel "{parts}" --poses "{pose}" --ctf "{ctf}" '
           f'--ind "{ind_pkl}" -o "{outdir}" -b {int(batch_size)} '
           f'--reg-weight {float(reg_weight)}')
    if os.environ["BP_UNINVERT"] == "1":
        cmd += " --uninvert-data"
    _n = len(np.asarray(utils.load_pkl(ind_pkl)))
    if lazy == "always":
        _use_lazy = True
    elif lazy == "never":
        _use_lazy = False
    else:
        _use_lazy = _eager_gb(_n) * 1e9 > _budget
    if _use_lazy:
        cmd += " --lazy"
    print(f"   {_n:,} particles -> {_eager_gb(_n):.1f} GB eager; "
          f"{'--lazy (streaming from disk)' if _use_lazy else 'loading into RAM'}")
    if int(first_n) > 0:
        cmd += f" --first {int(first_n)}"
    if os.environ["BP_DATADIR"]:
        cmd += f' --datadir "{os.environ["BP_DATADIR"]}"'

    print(f"\n[{i}/{len(todo)}] {name}")
    print("$", cmd, "\n" + "-" * 70)
    t0 = time.time()
    get_ipython().system(cmd)
    _rc = get_ipython().user_ns.get("_exit_code", 0)
    if _rc:
        raise RuntimeError(f"{name} failed (exit {_rc}) — see the log above. "
                           f"Fix it and re-run; finished sets will be skipped.")
    if not _done(outdir):
        raise RuntimeError(f"{name}: command reported success but wrote no backproject.mrc")
    dt = time.time() - t0
    shutil.copytree(outdir, os.path.join(drive_root, name), dirs_exist_ok=True)
    print("-" * 70 + f"\n✅ {name} in {dt/60:.1f} min → mirrored to Drive")

print("\n" + "=" * 70)
print(f"✅ All {len(sets)} set(s) reconstructed in {(time.time()-t_all)/60:.1f} min")
print(f"   local : {bp_root}\n   Drive : {drive_root}")

## 5 · Compare the reconstructions

**5.1** tabulates each subset's own half-map resolution — this is the honest per-subset number,
read from the phase-randomisation-corrected FSC that `backproject_voxel` already computed.

**5.2** builds the pairwise FSC matrix *between* subsets. Read this one carefully: an FSC between
two different subsets' maps is a **similarity** curve, not a resolution. Two subsets of the same
underlying structure correlate out to roughly their individual half-map resolutions, so the
useful signal is any pair that falls off markedly *sooner* than that.

**5.3** shows central slices and difference maps against the mean of all subsets — the most direct
way to see mass appearing or disappearing.

In [ ]:
#@title 5.1 · Per-subset resolution table { display-mode: "form" }
#@markdown FSC threshold for the reported resolution.
threshold = 0.143  #@param [0.143, 0.5] {type:"raw"}
#@markdown Which of the FSC curves to read (`Corrected` = phase-randomisation corrected, tight mask).
curve = "Corrected"  #@param ["Corrected", "Tight", "Loose", "Spherical", "NoMask"]

import os, json
import numpy as np
from cryodrgn import utils
from cryodrgn.mrcfile import parse_mrc

sets = json.loads(os.environ["BP_SETS"])
bp_root = os.environ["BP_ROOT"]

def _apix(d):
    _, hdr = parse_mrc(os.path.join(d, "backproject.mrc"))
    return float(hdr.apix)

def _res(d, col, thr):
    """Resolution (A) where the FSC curve last drops below `thr`."""
    f = os.path.join(d, "fsc-vals.txt")
    if not os.path.exists(f):
        return None
    tab = np.genfromtxt(f, names=True)
    if col not in tab.dtype.names:
        return None
    px, fsc = tab["pixres"], tab[col]
    below = np.where(fsc < thr)[0]
    if below.size == 0:                     # never drops below -> resolution-limited by Nyquist
        return _apix(d) / px[-1]
    i = below[0]
    if i == 0:
        return None
    # linear interpolation between the bracketing shells
    x0, x1, y0, y1 = px[i-1], px[i], fsc[i-1], fsc[i]
    xc = x0 + (y0 - thr) * (x1 - x0) / (y0 - y1) if y0 != y1 else x1
    return _apix(d) / xc

rows = []
for name, ind_pkl in sorted(sets.items()):
    d = os.path.join(bp_root, name)
    if not os.path.exists(os.path.join(d, "backproject.mrc")):
        continue
    n = len(np.asarray(utils.load_pkl(ind_pkl)))
    rows.append((name, n, _apix(d), _res(d, curve, float(threshold))))

if not rows:
    raise FileNotFoundError(f"No reconstructions under {bp_root} — run 4.1 first.")

w = max(len(r[0]) for r in rows)
print(f"{'set'.ljust(w)}  {'N':>9}  {'A/px':>6}  {curve} FSC={threshold}")
print("-" * (w + 34))
for name, n, apix, res in sorted(rows, key=lambda r: (r[3] is None, r[3] or 0)):
    print(f"{name.ljust(w)}  {n:>9,}  {apix:>6.3f}  "
          + (f"{res:>8.2f} A" if res else "       n/a"))

ns = np.array([r[1] for r in rows], float)
rs = np.array([r[3] if r[3] else np.nan for r in rows], float)
ok = ~np.isnan(rs)
if ok.sum() >= 3:
    # resolution should scale roughly as N^(-1/4) at fixed structure; a strong fit means
    # the differences are explained by particle count alone
    c = np.corrcoef(np.log(ns[ok]), np.log(rs[ok]))[0, 1]
    print("-" * (w + 34))
    print(f"corr(log N, log resolution) = {c:+.3f}")
    if c < -0.7:
        print("  → resolution tracks particle count closely; differences between these subsets\n"
              "    are mostly a counting effect, not a structural one. Equalise sizes (3.2).")
    else:
        print("  → resolution is NOT explained by particle count alone; worth looking at 5.2/5.3.")

In [ ]:
#@title 5.2 · Pairwise FSC matrix between subsets { display-mode: "form" }
#@markdown Resolution (A) at which each *pair* of subset maps decorrelates.
threshold = 0.5  #@param [0.143, 0.5] {type:"raw"}
#@markdown Apply a loose mask built from the mean map before comparing.
use_mask = True  #@param {type:"boolean"}

import os, json, itertools
import numpy as np
import matplotlib.pyplot as plt
import torch
from cryodrgn.mrcfile import parse_mrc
from cryodrgn.masking import cosine_dilation_mask
from cryodrgn.commands_utils.fsc import get_fsc_curve

sets = json.loads(os.environ["BP_SETS"])
bp_root = os.environ["BP_ROOT"]
names, vols = [], []
for name in sorted(sets):
    f = os.path.join(bp_root, name, "backproject.mrc")
    if os.path.exists(f):
        v, hdr = parse_mrc(f)
        names.append(name)
        vols.append(torch.tensor(np.asarray(v, dtype=np.float32)))
if len(vols) < 2:
    raise RuntimeError("Need at least two reconstructions — run 4.1 first.")
apix = float(hdr.apix)

mask = None
if use_mask:
    mean_vol = torch.stack(vols).mean(0).numpy()
    mask = torch.tensor(np.asarray(cosine_dilation_mask(
        mean_vol, dilation=25, edge_dist=15, apix=apix, verbose=False), dtype=np.float32))

n = len(names)
M = np.full((n, n), np.nan)
for i, j in itertools.combinations(range(n), 2):
    df = get_fsc_curve(vols[i], vols[j], initial_mask=mask)
    px, fsc = df.pixres.values, df.fsc.values
    below = np.where(fsc < float(threshold))[0]
    if below.size == 0 or below[0] == 0:
        r = apix / px[-1] if below.size == 0 else np.nan
    else:
        k = below[0]
        x0, x1, y0, y1 = px[k-1], px[k], fsc[k-1], fsc[k]
        xc = x0 + (y0 - float(threshold)) * (x1 - x0) / (y0 - y1) if y0 != y1 else x1
        r = apix / xc
    M[i, j] = M[j, i] = r
np.fill_diagonal(M, np.nan)

fig, ax = plt.subplots(figsize=(1 + 0.62 * n, 0.9 + 0.55 * n))
im = ax.imshow(M, cmap="viridis_r")
ax.set_xticks(range(n)); ax.set_xticklabels(names, rotation=90, fontsize=7)
ax.set_yticks(range(n)); ax.set_yticklabels(names, fontsize=7)
for i in range(n):
    for j in range(n):
        if not np.isnan(M[i, j]):
            ax.text(j, i, f"{M[i,j]:.1f}", ha="center", va="center", fontsize=6,
                    color="white" if M[i, j] > np.nanmedian(M) else "black")
ax.set_title(f"pairwise map-map FSC={threshold} (Å)\nlower = decorrelates sooner = more different",
             fontsize=9)
fig.colorbar(im, ax=ax, shrink=0.8, label="Å")
plt.tight_layout(); plt.show()

off = M[~np.isnan(M)]
print(f"pairwise resolution: median {np.median(off):.2f} Å, range {off.min():.2f}–{off.max():.2f} Å")
print(f"spread / median = {(off.max()-off.min())/np.median(off)*100:.1f}%")
if (off.max() - off.min()) / np.median(off) < 0.15:
    print("→ all pairs decorrelate at essentially the same point: no evidence that these\n"
          "  subsets differ structurally at this box size.")
else:
    i, j = np.unravel_index(np.nanargmin(M), M.shape)
    print(f"→ most dissimilar pair: {names[i]} vs {names[j]} ({M[i,j]:.2f} Å) — inspect in 5.3.")

In [ ]:
#@title 5.3 · Central slices and difference maps { display-mode: "form" }
#@markdown Difference is computed against the mean of all subset maps, after normalising each
#@markdown map to zero mean / unit standard deviation inside the mask (otherwise differing
#@markdown particle counts alone produce a difference).
sigma_clip = 4.0  #@param {type:"number"}
#@markdown Slice axis for the montage.
axis = "z"  #@param ["x", "y", "z"]

import os, json
import numpy as np
import matplotlib.pyplot as plt
from cryodrgn.mrcfile import parse_mrc
from cryodrgn.masking import cosine_dilation_mask

sets = json.loads(os.environ["BP_SETS"])
bp_root = os.environ["BP_ROOT"]
names, vols = [], []
for name in sorted(sets):
    f = os.path.join(bp_root, name, "backproject.mrc")
    if os.path.exists(f):
        v, hdr = parse_mrc(f)
        names.append(name); vols.append(np.asarray(v, dtype=np.float32))
if not vols:
    raise FileNotFoundError("No reconstructions found — run 4.1 first.")
apix = float(hdr.apix)

V = np.stack(vols)
mask = np.asarray(cosine_dilation_mask(V.mean(0), dilation=25, edge_dist=15,
                                       apix=apix, verbose=False), dtype=np.float32)
sel = mask > 0.5
# normalise inside the mask so subset size does not masquerade as a density difference
V = np.stack([(v - v[sel].mean()) / (v[sel].std() + 1e-9) for v in V])
mean_vol = V.mean(0)
D = V.shape[1]
ax_i = {"x": 0, "y": 1, "z": 2}[axis]
def _slice(vol):
    return np.take(vol, D // 2, axis=ax_i)

n = len(names)
fig, axes = plt.subplots(2, n, figsize=(1.9 * n, 4.2), squeeze=False)
dmax = np.percentile(np.abs(V - mean_vol), 99.5)
for k, (name, v) in enumerate(zip(names, V)):
    s = _slice(v)
    axes[0, k].imshow(s, cmap="gray")
    axes[0, k].set_title(name, fontsize=7)
    axes[0, k].axis("off")
    axes[1, k].imshow(_slice(v - mean_vol), cmap="bwr", vmin=-dmax, vmax=dmax)
    axes[1, k].axis("off")
axes[0, 0].set_ylabel("map", fontsize=8)
axes[1, 0].set_ylabel("− mean", fontsize=8)
fig.suptitle(f"central {axis}-slice (top) and difference from the mean map (bottom)  ·  "
             f"{apix:.2f} Å/px", fontsize=9)
plt.tight_layout(); plt.show()

print(f"{'set':<24} {'max |Δ| (σ)':>12}  {'voxels >' + str(sigma_clip) + 'σ':>14}")
print("-" * 54)
for name, v in zip(names, V):
    d = (v - mean_vol)[sel]
    print(f"{name:<24} {np.abs(d).max():>12.2f}  {int((np.abs(d) > sigma_clip).sum()):>14,}")
print("\nA subset that genuinely carries extra or missing mass shows a compact, contiguous\n"
      "red or blue blob in its difference panel. Speckle spread over the whole particle is\n"
      "noise from a smaller subset, not structure.")

In [ ]:
#@title 6.1 · Zip the reconstructions and download { display-mode: "form" }
#@markdown Volumes are already mirrored to Drive by 4.1. This bundles them for local
#@markdown inspection in ChimeraX.
include_half_maps = False  #@param {type:"boolean"}

import os, shutil, glob
bp_root = os.environ["BP_ROOT"]
WORK_DIR = os.environ["BP_WORK_DIR"]
stage = os.path.join(WORK_DIR, "backproject_bundle")
shutil.rmtree(stage, ignore_errors=True)
os.makedirs(stage)

n = 0
for d in sorted(glob.glob(os.path.join(bp_root, "*"))):
    if not os.path.isdir(d):
        continue
    name = os.path.basename(d)
    for fn in ("backproject.mrc", "fsc-vals.txt", "fsc-plot.png"):
        p = os.path.join(d, fn)
        if os.path.exists(p):
            ext = os.path.splitext(fn)[1]
            shutil.copyfile(p, os.path.join(stage, f"{name}{'' if fn.startswith('backproject') else '_fsc'}{ext}"))
    if include_half_maps:
        for fn in ("half_map_a.mrc", "half_map_b.mrc"):
            p = os.path.join(d, fn)
            if os.path.exists(p):
                shutil.copyfile(p, os.path.join(stage, f"{name}_{fn}"))
    n += 1

zip_base = os.path.join(WORK_DIR, "backproject_volumes")
shutil.make_archive(zip_base, "zip", stage)
size = os.path.getsize(zip_base + ".zip") / 2**20
shutil.copyfile(zip_base + ".zip", os.path.join(os.environ["BP_DRIVE_DIR"], "backproject_volumes.zip"))
print(f"✅ {n} reconstruction(s), {size:.1f} MB")
print(f"   Drive : {os.environ['BP_DRIVE_DIR']}/backproject_volumes.zip")
try:
    from google.colab import files
    files.download(zip_base + ".zip")
except Exception as e:
    print(f"   (browser download unavailable: {e})")

---
### Notes

**Reading the comparison.** The question "are these subsets structurally different?" is not
answered by their individual resolutions — those are dominated by particle count. It is answered
by 5.2 and 5.3 on **size-equalised** subsets. If every pair in 5.2 decorrelates at the same point
and 5.3 shows only speckle, the subsets are the same structure.

**Box size sets the ceiling.** A 128-px box at ~2.6 Å/px has a Nyquist limit of 5.2 Å, which is
ample for seeing a domain-sized mass appear or disappear and useless for anything finer. Your
`pose.pkl` and `ctf.pkl` work unchanged at *any* box size — poses are stored as a fraction of the
box (`pose.py:127-132`) and `ctf.py:160-162` rescales the pixel size — so you can point cell 2.1
at a larger stack with no re-parsing. The cost is I/O, not bookkeeping.

**`--first` interacts with `--ind`.** In 4.1, `first_n` caps each subset *after* index filtering,
so it takes the first N of the selected particles. Since the index files are sorted, that is the
N lowest stack positions, not a random draw — fine as a smoke test, not as a size-equaliser. Use
3.2 for that.